In [1]:
# ============================================================================
# 🧠 Subject Embedding 可行性分析完整框架
# 
# 目标：基于数据特征判断是否应该使用Subject Embedding方法
# 针对：脑体素分类任务，37个训练受试者 + 1个测试受试者，341维特征，102脑区分类
# ============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import f_oneway, kruskal
import h5py
import time
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.size'] = 12
plt.rcParams['figure.figsize'] = (12, 8)


In [2]:
import scipy.io
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, confusion_matrix, balanced_accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import json
from collections import Counter
import itertools


In [3]:

# ============================================================================
# 第一部分：数据预处理（保持Alex的原始逻辑）
# ============================================================================

# 设置随机种子确保可重现性

# 设置设备

# 📁 修改：设置保存路径
export_path = './enhanced_alexs_cb_focal_results/'
os.makedirs(export_path, exist_ok=True)
os.makedirs(os.path.join(export_path, 'visualizations'), exist_ok=True)
os.makedirs(os.path.join(export_path, 'gamma_comparison'), exist_ok=True)

# 📊 数据加载和预处理
print("📂 加载数据...")
f = h5py.File('/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat','r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

train_data = arrays['data'].transpose()
train_region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()
print(f"原始数据形状: {train_data.shape}")
print(f"原始标签形状: {train_region.shape}")
print(f"prob_idx形状: {prob_idx.shape}")

del arrays, f

# 创建更科学的数据分割
test_indices = np.where(prob_idx ==38)[0]
train_val_indices = np.where(prob_idx != 38)[0]

test_data = train_data[test_indices, :]
test_labels = train_region[test_indices, :]
train_val_data = train_data[train_val_indices, :]
train_val_labels = train_region[train_val_indices, :]

print(f"测试集形状: {test_data.shape}")
print(f"训练+验证集形状: {train_val_data.shape}")

# 进一步分割训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    train_val_data, train_val_labels, 
    test_size=0.2, random_state=42, stratify=np.argmax(train_val_labels, axis=1)
)

print(f"最终训练集形状: {X_train.shape}")
print(f"最终验证集形状: {X_val.shape}")
print(f"最终测试集形状: {test_data.shape}")

del train_data, train_region, prob_idx, train_val_data, train_val_labels, test_indices, train_val_indices

# 标准化
print("📊 应用标准化...")
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(test_data)

print("✅ 数据预处理完成")

📂 加载数据...
原始数据形状: (6968700, 341)
原始标签形状: (6968700, 102)
prob_idx形状: (6968700, 1)
测试集形状: (180025, 341)
训练+验证集形状: (6788675, 341)
最终训练集形状: (5430940, 341)
最终验证集形状: (1357735, 341)
最终测试集形状: (180025, 341)
📊 应用标准化...
✅ 数据预处理完成


In [ ]:
class SubjectEmbeddingAnalyzer:
    """
    Subject Embedding 可行性分析器
    
    完整分析流程：
    Phase 1: 受试者间差异分析
    Phase 2: 受试者可分离性评估  
    Phase 3: Embedding适配性评估
    Phase 4: 决策建议生成
    """
    
    def __init__(self, save_path='./subject_embedding_analysis/'):
        self.save_path = save_path
        os.makedirs(save_path, exist_ok=True)
        os.makedirs(os.path.join(save_path, 'visualizations'), exist_ok=True)
        
        # 分析结果存储
        self.analysis_results = {}
        self.decision_scores = {}
        
        print("🧠 Subject Embedding 可行性分析器初始化完成")
        print(f"📁 结果保存路径: {save_path}")

    def prepare_data_with_subjects_precise(self, X_train, y_train, X_val, y_val, X_test, test_labels, 
                                     prob_idx_path='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'):
        """
        精确重建受试者ID映射
        """
        print("\n" + "="*80)
        print("📊 Phase 0: 精确数据准备与受试者ID重建")
        print("="*80)
        
        # 1. 加载完整的原始数据来重建映射
        print("🔄 加载原始数据进行精确映射...")
        try:
            f = h5py.File(prob_idx_path, 'r')
            original_data = np.array(f['data']).transpose()
            original_region = np.array(f['region']).transpose() 
            prob_idx = np.array(f['prob_idx']).transpose().flatten()
            f.close()
            
            print(f"✅ 原始数据加载成功")
            print(f"  - 原始数据形状: {original_data.shape}")
            print(f"  - prob_idx形状: {prob_idx.shape}")
            
        except Exception as e:
            print(f"❌ 无法加载原始文件，使用备选方案: {e}")
            return self._fallback_subject_mapping(X_train, y_train, X_val, y_val, X_test, test_labels)
        
        # 2. 应用相同的标准化
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        original_data_scaled = scaler.fit_transform(original_data)
        
        # 3. 重现相同的train_test_split过程
        test_indices = np.where(prob_idx == 38)[0]  
        train_val_indices = np.where(prob_idx != 38)[0]
        
        original_test_data = original_data_scaled[test_indices]
        original_test_labels = original_region[test_indices]
        original_train_val_data = original_data_scaled[train_val_indices]
        original_train_val_labels = original_region[train_val_indices]
        original_train_val_subjects = prob_idx[train_val_indices]
        
        print(f"🔍 原始分割结果:")
        print(f"  - 训练+验证: {len(train_val_indices)}")
        print(f"  - 测试: {len(test_indices)}")
        
        # 4. 重现train_test_split来建立精确映射
        from sklearn.model_selection import train_test_split
        
        # 使用相同的参数重现分割
        X_train_orig, X_val_orig, y_train_orig, y_val_orig, subjects_train_orig, subjects_val_orig = train_test_split(
            original_train_val_data, 
            original_train_val_labels,
            original_train_val_subjects,
            test_size=0.2, 
            random_state=42,  # 关键：使用相同的随机种子
            stratify=np.argmax(original_train_val_labels, axis=1)
        )
        
        print(f"🔍 重现分割结果:")
        print(f"  - 训练: {len(X_train_orig)}")
        print(f"  - 验证: {len(X_val_orig)}")
        
        # 5. 验证数据一致性
        train_match = np.allclose(X_train_orig, X_train, rtol=1e-5, atol=1e-8)
        val_match = np.allclose(X_val_orig, X_val, rtol=1e-5, atol=1e-8)
        test_match = np.allclose(original_test_data, X_test, rtol=1e-5, atol=1e-8)
        
        print(f"🔍 数据一致性验证:")
        print(f"  - 训练集匹配: {train_match}")
        print(f"  - 验证集匹配: {val_match}")
        print(f"  - 测试集匹配: {test_match}")
        
        if train_match and val_match and test_match:
            print("✅ 数据完全匹配，使用精确的受试者映射")
            train_subject_ids = subjects_train_orig
            val_subject_ids = subjects_val_orig
            test_subject_ids = np.full(len(X_test), 38)  
            
        else:
            print("⚠️ 数据不完全匹配，使用近似映射方法")
            return self._approximate_subject_mapping(X_train, y_train, X_val, y_val, X_test, test_labels, 
                                                    original_train_val_data, original_train_val_subjects)
        
        # 6. 存储精确映射的数据
        available_subjects = np.unique(np.concatenate([subjects_train_orig, subjects_val_orig]))
        
        self.data = {
            'X_train': X_train, 'y_train': y_train, 'subjects_train': train_subject_ids,
            'X_val': X_val, 'y_val': y_val, 'subjects_val': val_subject_ids,
            'X_test': X_test, 'y_test': test_labels, 'subjects_test': test_subject_ids,
            'available_subjects': available_subjects,
            'subject_counts': {subj: np.sum(np.concatenate([train_subject_ids, val_subject_ids]) == subj) 
                              for subj in available_subjects}
        }
        
        print(f"✅ 精确受试者映射完成")
        print(f"  - 训练受试者: {len(np.unique(train_subject_ids))} 个")
        print(f"  - 验证受试者: {len(np.unique(val_subject_ids))} 个") 
        print(f"  - 受试者ID范围: {available_subjects}")
        
        return self.data

    def _approximate_subject_mapping(self, X_train, y_train, X_val, y_val, X_test, test_labels,
                                    original_train_val_data, original_train_val_subjects):
        """
        当精确映射失败时，使用特征相似性进行近似映射
        """
        print("🔄 使用特征相似性进行近似受试者映射...")
        
        # 1. 计算原始数据中每个受试者的平均特征
        available_subjects = np.unique(original_train_val_subjects)
        subject_profiles = {}
        
        for subject_id in available_subjects:
            subject_mask = original_train_val_subjects == subject_id
            if np.sum(subject_mask) > 100:  # 确保有足够样本
                subject_profile = np.mean(original_train_val_data[subject_mask], axis=0)
                subject_profiles[subject_id] = subject_profile
        
        print(f"📊 构建了 {len(subject_profiles)} 个受试者特征档案")
        
        # 2. 为训练和验证数据分配受试者
        def assign_subjects_by_similarity(data_split, split_name):
            print(f"🔍 为{split_name}分配受试者...")
            
            # 随机采样一些体素来代表这个数据分割
            n_samples = min(10000, len(data_split))
            sample_indices = np.random.choice(len(data_split), n_samples, replace=False)
            sample_data = data_split[sample_indices]
            
            # 计算样本与每个受试者档案的相似性
            subject_assignments = []
            
            for i, voxel_features in enumerate(sample_data):
                similarities = {}
                for subject_id, profile in subject_profiles.items():
                    # 使用余弦相似性
                    similarity = np.dot(voxel_features, profile) / (
                        np.linalg.norm(voxel_features) * np.linalg.norm(profile) + 1e-8
                    )
                    similarities[subject_id] = similarity
                
                # 选择最相似的受试者
                best_subject = max(similarities, key=similarities.get)
                subject_assignments.append(best_subject)
            
            # 基于采样结果，为整个数据分割分配受试者
            # 使用采样的受试者分布作为概率
            from collections import Counter
            subject_counts = Counter(subject_assignments)
            subject_probs = np.array([subject_counts.get(subj, 0) for subj in available_subjects])
            subject_probs = subject_probs / subject_probs.sum()
            
            # 为整个数据分割分配受试者
            full_assignments = np.random.choice(
                available_subjects, 
                size=len(data_split), 
                p=subject_probs
            )
            
            print(f"✅ {split_name}受试者分配完成，涉及 {len(np.unique(full_assignments))} 个受试者")
            return full_assignments
        
        # 分配受试者
        train_subject_ids = assign_subjects_by_similarity(X_train, "训练集")
        val_subject_ids = assign_subjects_by_similarity(X_val, "验证集")
        test_subject_ids = np.full(len(X_test), 38) 
        
        # 存储数据
        self.data = {
            'X_train': X_train, 'y_train': y_train, 'subjects_train': train_subject_ids,
            'X_val': X_val, 'y_val': y_val, 'subjects_val': val_subject_ids,
            'X_test': X_test, 'y_test': test_labels, 'subjects_test': test_subject_ids,
            'available_subjects': available_subjects,
            'subject_counts': {subj: np.sum(np.concatenate([train_subject_ids, val_subject_ids]) == subj) 
                              for subj in available_subjects}
        }
        
        print("✅ 基于相似性的近似映射完成")
        return self.data

    def _fallback_subject_mapping(self, X_train, y_train, X_val, y_val, X_test, test_labels):
        """
        备选方案：承认无法精确映射，进行有限的分析
        """
        print("⚠️ 使用备选分析方案...")
        print("💡 将进行不依赖精确受试者映射的分析")
        
        # 使用基于数据块的伪受试者分组
        # 假设数据是按某种顺序排列的，尝试分块
        n_pseudo_subjects = 20  # 创建20个伪受试者组
        
        block_size_train = len(X_train) // n_pseudo_subjects
        block_size_val = len(X_val) // n_pseudo_subjects
        
        train_subject_ids = np.repeat(range(n_pseudo_subjects), block_size_train)
        # 处理剩余的样本
        remaining_train = len(X_train) - len(train_subject_ids)
        if remaining_train > 0:
            train_subject_ids = np.concatenate([
                train_subject_ids, 
                np.full(remaining_train, n_pseudo_subjects-1)
            ])
        
        val_subject_ids = np.repeat(range(n_pseudo_subjects), block_size_val)
        remaining_val = len(X_val) - len(val_subject_ids)
        if remaining_val > 0:
            val_subject_ids = np.concatenate([
                val_subject_ids,
                np.full(remaining_val, n_pseudo_subjects-1)
            ])
        
        test_subject_ids = np.full(len(X_test), n_pseudo_subjects)  # 测试组单独编号
        
        available_subjects = np.arange(n_pseudo_subjects + 1)
        
        self.data = {
            'X_train': X_train, 'y_train': y_train, 'subjects_train': train_subject_ids,
            'X_val': X_val, 'y_val': y_val, 'subjects_val': val_subject_ids,
            'X_test': X_test, 'y_test': test_labels, 'subjects_test': test_subject_ids,
            'available_subjects': available_subjects,
            'subject_counts': {subj: np.sum(np.concatenate([train_subject_ids, val_subject_ids]) == subj) 
                              for subj in available_subjects[:-1]},  # 排除测试组
            'mapping_method': 'fallback_blocks'
        }
        
        print(f"✅ 备选映射完成，创建了 {n_pseudo_subjects} 个伪受试者组")
        print("⚠️ 注意：分析结果仅供参考，建议获取真实的受试者映射")
        
        return self.data
    
    def prepare_data_with_subjects(self, X_train, y_train, X_val, y_val, X_test, test_labels, 
                             prob_idx_path='/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/DATA/TRAIN38.mat'):
        """
        数据准备：尝试多种方法重建准确的受试者ID映射
        """
        # 方案1：尝试精确映射
        try:
            return self.prepare_data_with_subjects_precise(
                X_train, y_train, X_val, y_val, X_test, test_labels, prob_idx_path
            )
        except Exception as e:
            print(f"⚠️ 精确映射失败: {e}")
            
            # 方案2：尝试相似性映射  
            try:
                return self._approximate_subject_mapping(
                    X_train, y_train, X_val, y_val, X_test, test_labels, None, None
                )
            except Exception as e:
                print(f"⚠️ 相似性映射失败: {e}")
                
                # 方案3：备选方案
                return self._fallback_subject_mapping(
                    X_train, y_train, X_val, y_val, X_test, test_labels
                )

    # 添加其他所有方法（phase1, phase2, phase3等）
    # 为了简洁，这里省略了其他方法的代码，但它们应该和之前提供的代码一样
    
    def phase1_subject_differences_analysis(self):
        """
        Phase 1: 受试者间差异本质分析
        
        1.1 统计分布差异诊断
        1.2 特征空间变异模式分析
        """
        print("\n" + "="*80)
        print("📊 Phase 1: 受试者间差异本质分析")
        print("="*80)
        
        # 1.1 统计分布差异诊断
        print("\n🔍 1.1 计算受试者统计特征...")
        
        subjects = self.data['available_subjects']
        n_features = self.data['X_train'].shape[1]
        
        # 为每个受试者计算统计量
        subject_stats = {
            'means': np.zeros((len(subjects), n_features)),
            'stds': np.zeros((len(subjects), n_features)),
            'skews': np.zeros((len(subjects), n_features)),
            'kurts': np.zeros((len(subjects), n_features)),
            'sample_counts': np.zeros(len(subjects))
        }
        
        for i, subject_id in enumerate(subjects):
            # 训练集中该受试者的数据
            subject_mask = self.data['subjects_train'] == subject_id
            if np.sum(subject_mask) > 0:
                subject_data = self.data['X_train'][subject_mask]
                
                subject_stats['means'][i] = np.mean(subject_data, axis=0)
                subject_stats['stds'][i] = np.std(subject_data, axis=0)
                subject_stats['skews'][i] = stats.skew(subject_data, axis=0)
                subject_stats['kurts'][i] = stats.kurtosis(subject_data, axis=0)
                subject_stats['sample_counts'][i] = len(subject_data)
            
            # 处理验证集
            val_mask = self.data['subjects_val'] == subject_id
            if np.sum(val_mask) > 0:
                val_data = self.data['X_val'][val_mask]
                subject_stats['sample_counts'][i] += len(val_data)
        
        self.analysis_results['subject_stats'] = subject_stats
        
        print(f"✅ 受试者统计特征计算完成")
        print(f"  - 平均每受试者样本量: {np.mean(subject_stats['sample_counts']):.0f}")
        print(f"  - 样本量范围: [{np.min(subject_stats['sample_counts']):.0f}, {np.max(subject_stats['sample_counts']):.0f}]")
        
        # 1.2 受试者间相似性分析
        print("\n🔍 1.2 分析受试者间相似性...")
        
        # 计算受试者间距离（基于均值向量）
        subject_means = subject_stats['means']
        
        # 欧氏距离矩阵
        distance_matrix = squareform(pdist(subject_means, metric='euclidean'))
        
        # 相关性矩阵
        correlation_matrix = np.corrcoef(subject_means)
        
        # 层次聚类
        linkage_matrix = linkage(subject_means, method='ward')
        
        self.analysis_results['subject_similarity'] = {
            'distance_matrix': distance_matrix,
            'correlation_matrix': correlation_matrix,
            'linkage_matrix': linkage_matrix
        }
        
        print(f"✅ 受试者相似性分析完成")
        print(f"  - 平均受试者间距离: {np.mean(distance_matrix[np.triu_indices_from(distance_matrix, k=1)]):.4f}")
        print(f"  - 平均受试者间相关性: {np.mean(correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)]):.4f}")
        
        # 1.3 特征变异模式分析
        print("\n🔍 1.3 分析特征变异模式...")
        
        # 对每个特征计算受试者间的F统计量
        feature_f_stats = np.zeros(n_features)
        feature_p_values = np.zeros(n_features)
        
        for feat_idx in range(n_features):
            # 收集每个受试者该特征的均值
            subject_feature_values = subject_stats['means'][:, feat_idx]
            
            # 计算该特征的受试者间方差 vs 受试者内方差
            # 这里简化为受试者均值的方差
            between_subject_var = np.var(subject_feature_values)
            
            # 计算F统计量的近似
            # 使用受试者均值的标准差作为指标
            feature_f_stats[feat_idx] = between_subject_var
        
        # 归一化F统计量
        feature_f_stats_norm = feature_f_stats / np.mean(feature_f_stats)
        
        # 识别高变异特征（受试者间差异大的特征）
        high_variation_threshold = np.percentile(feature_f_stats_norm, 90)
        low_variation_threshold = np.percentile(feature_f_stats_norm, 10)
        
        high_variation_features = np.where(feature_f_stats_norm > high_variation_threshold)[0]
        low_variation_features = np.where(feature_f_stats_norm < low_variation_threshold)[0]
        
        # PCA分析受试者差异模式
        pca = PCA()
        subject_pca_result = pca.fit_transform(subject_means)
        
        self.analysis_results['feature_variation'] = {
            'f_stats': feature_f_stats_norm,
            'high_variation_features': high_variation_features,
            'low_variation_features': low_variation_features,
            'pca_result': subject_pca_result,
            'pca_explained_variance': pca.explained_variance_ratio_,
            'pca_cumulative_variance': np.cumsum(pca.explained_variance_ratio_)
        }
        
        print(f"✅ 特征变异分析完成")
        print(f"  - 高变异特征数量: {len(high_variation_features)} ({len(high_variation_features)/n_features*100:.1f}%)")
        print(f"  - 低变异特征数量: {len(low_variation_features)} ({len(low_variation_features)/n_features*100:.1f}%)")
        print(f"  - 前3个主成分解释方差: {np.sum(pca.explained_variance_ratio_[:3])*100:.1f}%")
        print(f"  - 前10个主成分解释方差: {np.sum(pca.explained_variance_ratio_[:10])*100:.1f}%")
        
        # 决策得分计算
        variance_explained_3pc = np.sum(pca.explained_variance_ratio_[:3])
        mean_correlation = np.mean(correlation_matrix[np.triu_indices_from(correlation_matrix, k=1)])
        
        # Phase 1 决策得分
        self.decision_scores['phase1'] = {
            'difference_significance': 1.0 if variance_explained_3pc > 0.6 else 0.5,
            'pattern_linearity': variance_explained_3pc,  # 越高越线性
            'subject_similarity': abs(mean_correlation),  # 相关性绝对值
            'feature_heterogeneity': len(high_variation_features) / n_features
        }
        
        print(f"\n📈 Phase 1 决策指标:")
        print(f"  - 差异显著性得分: {self.decision_scores['phase1']['difference_significance']:.3f}")
        print(f"  - 模式线性度: {self.decision_scores['phase1']['pattern_linearity']:.3f}")
        print(f"  - 受试者相似性: {self.decision_scores['phase1']['subject_similarity']:.3f}")
        print(f"  - 特征异质性: {self.decision_scores['phase1']['feature_heterogeneity']:.3f}")
    
    def phase2_subject_separability_analysis(self):
        """
        Phase 2: 受试者可分离性评估
        
        2.1 受试者识别难度测试
        2.2 脑区分类一致性分析
        """
        print("\n" + "="*80)
        print("📊 Phase 2: 受试者可分离性评估")
        print("="*80)
        
        # 2.1 受试者识别测试
        print("\n🔍 2.1 受试者识别难度测试...")
        
        # 准备受试者识别数据
        X_subject_id = self.data['X_train']
        y_subject_id = self.data['subjects_train']
        
        # 🔧 修复：确保受试者ID是整数类型
        y_subject_id = y_subject_id.astype(int)
        
        # 只保留样本量足够的受试者
        # 🔧 修复：处理可能的负数或超大数值
        max_subject_id = max(y_subject_id)
        min_subject_id = min(y_subject_id)
        
        if min_subject_id < 0:
            print(f"⚠️ 发现负数受试者ID，进行调整...")
            y_subject_id = y_subject_id - min_subject_id
            max_subject_id = max_subject_id - min_subject_id
        
        if max_subject_id > 10000:  # 防止内存溢出
            print(f"⚠️ 受试者ID过大 ({max_subject_id})，使用重新映射...")
            unique_subjects = np.unique(y_subject_id)
            subject_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_subjects)}
            y_subject_id = np.array([subject_mapping[old_id] for old_id in y_subject_id])
        
        try:
            subject_counts = np.bincount(y_subject_id)
            valid_subjects = np.where(subject_counts >= 1000)[0]  # 至少1000个样本
        except Exception as e:
            print(f"⚠️ bincount失败: {e}")
            # 备选方案：手动计算
            unique_subjects, counts = np.unique(y_subject_id, return_counts=True)
            valid_subjects = unique_subjects[counts >= 1000]
            subject_counts = dict(zip(unique_subjects, counts))
        
        if len(valid_subjects) > 0:
            # 筛选数据
            valid_mask = np.isin(y_subject_id, valid_subjects)
            X_subset = X_subject_id[valid_mask]
            y_subset = y_subject_id[valid_mask]
            
            # 重新编码受试者ID到连续范围
            subject_mapping = {old_id: new_id for new_id, old_id in enumerate(valid_subjects)}
            y_subset_remapped = np.array([subject_mapping[old_id] for old_id in y_subset])
            
            print(f"  - 有效受试者数量: {len(valid_subjects)}")
            print(f"  - 用于测试的样本数: {len(X_subset)}")
            
            # 交叉验证受试者识别
            classifier = LogisticRegression(random_state=42, max_iter=1000)
            cv_scores = cross_val_score(classifier, X_subset, y_subset_remapped, 
                                    cv=min(5, len(valid_subjects)), scoring='accuracy')
            
            subject_id_accuracy = np.mean(cv_scores)
            
            # 特征重要性分析（使用随机森林）
            rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
            rf_classifier.fit(X_subset, y_subset_remapped)
            feature_importance = rf_classifier.feature_importances_
            
            self.analysis_results['subject_identification'] = {
                'accuracy': subject_id_accuracy,
                'cv_scores': cv_scores,
                'feature_importance': feature_importance,
                'valid_subjects': valid_subjects,
                'n_valid_subjects': len(valid_subjects)
            }
            
            print(f"✅ 受试者识别测试完成")
            print(f"  - 识别准确率: {subject_id_accuracy:.3f} (随机基线: {1/len(valid_subjects):.3f})")
            print(f"  - 标准差: {np.std(cv_scores):.3f}")
            
        else:
            print("❌ 没有足够样本的受试者进行识别测试")
            self.analysis_results['subject_identification'] = {'accuracy': 0.0}
            subject_id_accuracy = 0.0
        
        # 2.2 脑区分类一致性分析
        print("\n🔍 2.2 脑区分类一致性分析...")
        
        # 转换one-hot标签到类别索引
        if len(self.data['y_train'].shape) > 1 and self.data['y_train'].shape[1] > 1:
            y_train_classes = np.argmax(self.data['y_train'], axis=1)
        else:
            y_train_classes = self.data['y_train'].flatten()
        
        unique_classes = np.unique(y_train_classes)
        n_classes = len(unique_classes)
        
        # 计算每个脑区在不同受试者间的一致性
        class_consistency = {}
        
        # 🔧 修复：确保受试者ID在这里也是整数
        subjects_train_int = self.data['subjects_train'].astype(int)
        
        for class_id in unique_classes[:20]:  # 只分析前20个类别，避免计算过久
            class_mask = y_train_classes == class_id
            
            if np.sum(class_mask) > 100:  # 确保有足够样本
                class_data = self.data['X_train'][class_mask]
                class_subjects = subjects_train_int[class_mask]
                
                # 计算每个受试者该脑区的均值特征
                subject_means_for_class = []
                subjects_with_class = []
                
                for subject_id in np.unique(class_subjects):
                    subject_class_mask = class_subjects == subject_id
                    if np.sum(subject_class_mask) >= 10:  # 至少10个样本
                        subject_mean = np.mean(class_data[subject_class_mask], axis=0)
                        subject_means_for_class.append(subject_mean)
                        subjects_with_class.append(subject_id)
                
                if len(subject_means_for_class) >= 3:  # 至少3个受试者
                    subject_means_array = np.array(subject_means_for_class)
                    
                    # 计算受试者间的变异系数
                    feature_cv = np.std(subject_means_array, axis=0) / (np.abs(np.mean(subject_means_array, axis=0)) + 1e-8)
                    mean_cv = np.mean(feature_cv)
                    
                    class_consistency[class_id] = {
                        'mean_cv': mean_cv,
                        'n_subjects': len(subjects_with_class),
                        'n_samples': np.sum(class_mask)
                    }
        
        self.analysis_results['class_consistency'] = class_consistency
        
        if class_consistency:
            avg_consistency = np.mean([info['mean_cv'] for info in class_consistency.values()])
            print(f"✅ 脑区一致性分析完成")
            print(f"  - 分析脑区数量: {len(class_consistency)}")
            print(f"  - 平均变异系数: {avg_consistency:.3f} (越低越一致)")
        else:
            print("❌ 脑区一致性分析失败")
            avg_consistency = 1.0
        
        # Phase 2 决策得分计算
        random_baseline = 1 / len(valid_subjects) if len(valid_subjects) > 0 else 0.1
        separability_score = max(0, min(1, (subject_id_accuracy - random_baseline) / (1 - random_baseline)))
        consistency_score = max(0, min(1, 1 / (1 + avg_consistency)))  # 变异系数越低，得分越高
        
        self.decision_scores['phase2'] = {
            'subject_separability': separability_score,
            'class_consistency': consistency_score,
            'identification_accuracy': subject_id_accuracy,
            'feature_competition_risk': separability_score  # 识别度越高，特征竞争风险越大
        }
        
        print(f"\n📈 Phase 2 决策指标:")
        print(f"  - 受试者可分离性: {self.decision_scores['phase2']['subject_separability']:.3f}")
        print(f"  - 脑区分类一致性: {self.decision_scores['phase2']['class_consistency']:.3f}")
        print(f"  - 特征竞争风险: {self.decision_scores['phase2']['feature_competition_risk']:.3f}")
    
    def phase3_embedding_adaptability_analysis(self):
        """
        Phase 3: Embedding适配性评估
        
        3.1 线性vs非线性差异模式检测
        3.2 样本量充足性评估
        """
        print("\n" + "="*80)
        print("📊 Phase 3: Embedding适配性评估")
        print("="*80)
        
        # 3.1 线性vs非线性模式检测
        print("\n🔍 3.1 线性vs非线性模式检测...")
        
        subject_means = self.analysis_results['subject_stats']['means']
        
        # PCA降维到2D
        pca_2d = PCA(n_components=2)
        subject_pca_2d = pca_2d.fit_transform(subject_means)
        
        # t-SNE降维到2D
        try:
            tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(subject_means)-1))
            subject_tsne_2d = tsne_2d.fit_transform(subject_means)
        except Exception as e:
            print(f"⚠️ t-SNE失败: {e}")
            subject_tsne_2d = subject_pca_2d  # 使用PCA作为后备
        
        # 比较PCA和t-SNE的结构相似性（简化版）
        # 计算距离矩阵的相关性
        pca_distances = squareform(pdist(subject_pca_2d))
        tsne_distances = squareform(pdist(subject_tsne_2d))
        
        structure_correlation = np.corrcoef(pca_distances.flatten(), tsne_distances.flatten())[0, 1]
        
        # 聚类分析
        n_clusters_range = range(2, min(8, len(subject_means)//2))
        silhouette_scores = []
        
        for n_clusters in n_clusters_range:
            kmeans = KMeans(n_clusters=n_clusters, random_state=42)
            cluster_labels = kmeans.fit_predict(subject_means)
            silhouette_avg = silhouette_score(subject_means, cluster_labels)
            silhouette_scores.append(silhouette_avg)
        
        best_n_clusters = n_clusters_range[np.argmax(silhouette_scores)] if silhouette_scores else 2
        best_silhouette = max(silhouette_scores) if silhouette_scores else 0
        
        self.analysis_results['linearity_analysis'] = {
            'pca_2d': subject_pca_2d,
            'tsne_2d': subject_tsne_2d,
            'structure_correlation': structure_correlation,
            'best_n_clusters': best_n_clusters,
            'best_silhouette': best_silhouette,
            'silhouette_scores': silhouette_scores
        }
        
        print(f"✅ 线性vs非线性分析完成")
        print(f"  - PCA vs t-SNE结构相关性: {structure_correlation:.3f}")
        print(f"  - 最优聚类数: {best_n_clusters}")
        print(f"  - 最佳轮廓系数: {best_silhouette:.3f}")
        
        # 3.2 样本量充足性评估
        print("\n🔍 3.2 样本量充足性评估...")
        
        sample_counts = self.analysis_results['subject_stats']['sample_counts']
        total_samples = np.sum(sample_counts)
        n_subjects = len(sample_counts)
        avg_samples_per_subject = np.mean(sample_counts)
        min_samples_per_subject = np.min(sample_counts)
        
        # 评估样本量是否充足
        sample_adequacy_score = 0
        
        # 评估每个受试者的样本量
        if avg_samples_per_subject >= 50000:
            sample_adequacy_score += 0.4
        elif avg_samples_per_subject >= 10000:
            sample_adequacy_score += 0.2
        
        # 评估受试者数量
        if n_subjects >= 30:
            sample_adequacy_score += 0.3
        elif n_subjects >= 20:
            sample_adequacy_score += 0.2
        
        # 评估总样本量
        if total_samples >= 1000000:
            sample_adequacy_score += 0.3
        elif total_samples >= 500000:
            sample_adequacy_score += 0.2
        
        self.analysis_results['sample_adequacy'] = {
            'total_samples': total_samples,
            'n_subjects': n_subjects,
            'avg_samples_per_subject': avg_samples_per_subject,
            'min_samples_per_subject': min_samples_per_subject,
            'adequacy_score': sample_adequacy_score
        }
        
        print(f"✅ 样本量评估完成")
        print(f"  - 总样本量: {total_samples:,}")
        print(f"  - 受试者数量: {n_subjects}")
        print(f"  - 平均每受试者样本量: {avg_samples_per_subject:.0f}")
        print(f"  - 样本充足性得分: {sample_adequacy_score:.3f}")
        
        # Phase 3 决策得分计算
        linearity_score = max(0, min(1, structure_correlation))  # 相关性越高越线性
        clustering_score = best_silhouette if best_silhouette > 0 else 0
        
        self.decision_scores['phase3'] = {
            'linearity': linearity_score,
            'clustering_quality': clustering_score,
            'sample_adequacy': sample_adequacy_score,
            'embedding_feasibility': (linearity_score + sample_adequacy_score) / 2
        }
        
        print(f"\n📈 Phase 3 决策指标:")
        print(f"  - 线性度: {self.decision_scores['phase3']['linearity']:.3f}")
        print(f"  - 聚类质量: {self.decision_scores['phase3']['clustering_quality']:.3f}")
        print(f"  - 样本充足性: {self.decision_scores['phase3']['sample_adequacy']:.3f}")
        print(f"  - Embedding可行性: {self.decision_scores['phase3']['embedding_feasibility']:.3f}")
    
    def generate_visualizations(self):
        """生成所有分析结果的可视化图表"""
        print("\n" + "="*80)
        print("📊 生成可视化图表")
        print("="*80)
        
        # 创建大图
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # 1. 受试者相似性热图
        ax = axes[0, 0]
        correlation_matrix = self.analysis_results['subject_similarity']['correlation_matrix']
        im = ax.imshow(correlation_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
        ax.set_title('受试者间相关性矩阵', fontweight='bold')
        ax.set_xlabel('受试者 ID')
        ax.set_ylabel('受试者 ID')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # 2. 层次聚类树状图
        ax = axes[0, 1]
        linkage_matrix = self.analysis_results['subject_similarity']['linkage_matrix']
        dendrogram(linkage_matrix, ax=ax, leaf_rotation=90)
        ax.set_title('受试者层次聚类', fontweight='bold')
        ax.set_xlabel('受试者 ID')
        ax.set_ylabel('距离')
        
        # 3. 特征变异分布
        ax = axes[0, 2]
        f_stats = self.analysis_results['feature_variation']['f_stats']
        ax.hist(f_stats, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        ax.axvline(np.percentile(f_stats, 90), color='red', linestyle='--', 
                  label=f'90th percentile: {np.percentile(f_stats, 90):.2f}')
        ax.axvline(np.percentile(f_stats, 10), color='green', linestyle='--',
                  label=f'10th percentile: {np.percentile(f_stats, 10):.2f}')
        ax.set_title('特征受试者间变异分布', fontweight='bold')
        ax.set_xlabel('标准化F统计量')
        ax.set_ylabel('特征数量')
        ax.legend()
        
        # 4. PCA累积方差解释
        ax = axes[0, 3]
        cumulative_variance = self.analysis_results['feature_variation']['pca_cumulative_variance']
        ax.plot(range(1, min(21, len(cumulative_variance)+1)), 
               cumulative_variance[:20], 'o-', linewidth=2, markersize=6)
        ax.axhline(0.8, color='red', linestyle='--', label='80%解释阈值')
        ax.axhline(0.9, color='orange', linestyle='--', label='90%解释阈值')
        ax.set_title('PCA累积方差解释', fontweight='bold')
        ax.set_xlabel('主成分数量')
        ax.set_ylabel('累积方差解释比例')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 5. 受试者识别准确率
        ax = axes[1, 0]
        if 'subject_identification' in self.analysis_results:
            cv_scores = self.analysis_results['subject_identification'].get('cv_scores', [0])
            accuracy = self.analysis_results['subject_identification'].get('accuracy', 0)
            n_subjects = self.analysis_results['subject_identification'].get('n_valid_subjects', 1)
            random_baseline = 1 / n_subjects
            
            ax.bar(['随机基线', '逻辑回归'], [random_baseline, accuracy], 
                  color=['gray', 'lightcoral'])
            ax.set_title('受试者识别准确率对比', fontweight='bold')
            ax.set_ylabel('准确率')
            
            # 添加数值标签
            ax.text(0, random_baseline + 0.01, f'{random_baseline:.3f}', 
                   ha='center', va='bottom')
            ax.text(1, accuracy + 0.01, f'{accuracy:.3f}', 
                   ha='center', va='bottom')
        
        # 6. 特征重要性（Top 20）
        ax = axes[1, 1]
        if 'subject_identification' in self.analysis_results and 'feature_importance' in self.analysis_results['subject_identification']:
            feature_importance = self.analysis_results['subject_identification']['feature_importance']
            top_features = np.argsort(feature_importance)[-20:]
            ax.barh(range(20), feature_importance[top_features], color='lightgreen')
            ax.set_title('Top 20 受试者识别特征重要性', fontweight='bold')
            ax.set_xlabel('重要性得分')
            ax.set_ylabel('特征索引')
            ax.set_yticks(range(20))
            ax.set_yticklabels(top_features)
        
        # 7. PCA vs t-SNE受试者分布对比
        ax = axes[1, 2]
        if 'linearity_analysis' in self.analysis_results:
            pca_2d = self.analysis_results['linearity_analysis']['pca_2d']
            scatter = ax.scatter(pca_2d[:, 0], pca_2d[:, 1], 
                               c=range(len(pca_2d)), cmap='viridis', s=50, alpha=0.7)
            ax.set_title('PCA - 受试者2D分布', fontweight='bold')
            ax.set_xlabel('PC1')
            ax.set_ylabel('PC2')
            plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        
        ax = axes[1, 3]
        if 'linearity_analysis' in self.analysis_results:
            tsne_2d = self.analysis_results['linearity_analysis']['tsne_2d']
            scatter = ax.scatter(tsne_2d[:, 0], tsne_2d[:, 1], 
                               c=range(len(tsne_2d)), cmap='viridis', s=50, alpha=0.7)
            ax.set_title('t-SNE - 受试者2D分布', fontweight='bold')
            ax.set_xlabel('t-SNE 1')
            ax.set_ylabel('t-SNE 2')
            plt.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
        
        # 8. 聚类质量评估
        ax = axes[2, 0]
        if 'linearity_analysis' in self.analysis_results and 'silhouette_scores' in self.analysis_results['linearity_analysis']:
            silhouette_scores = self.analysis_results['linearity_analysis']['silhouette_scores']
            n_clusters_range = range(2, 2 + len(silhouette_scores))
            ax.plot(n_clusters_range, silhouette_scores, 'o-', linewidth=2, markersize=8)
            best_n = self.analysis_results['linearity_analysis']['best_n_clusters']
            best_score = self.analysis_results['linearity_analysis']['best_silhouette']
            ax.axvline(best_n, color='red', linestyle='--', 
                      label=f'最优聚类数: {best_n}')
            ax.set_title('聚类质量评估 (轮廓系数)', fontweight='bold')
            ax.set_xlabel('聚类数量')
            ax.set_ylabel('轮廓系数')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 9. 脑区一致性分析
        ax = axes[2, 1]
        if 'class_consistency' in self.analysis_results and self.analysis_results['class_consistency']:
            class_consistency = self.analysis_results['class_consistency']
            class_ids = list(class_consistency.keys())
            cv_values = [class_consistency[cid]['mean_cv'] for cid in class_ids]
            
            ax.bar(range(len(class_ids)), cv_values, color='lightblue', alpha=0.7)
            ax.set_title('脑区受试者间一致性', fontweight='bold')
            ax.set_xlabel('脑区 ID')
            ax.set_ylabel('变异系数 (越低越一致)')
            ax.set_xticks(range(len(class_ids)))
            ax.set_xticklabels(class_ids, rotation=45)
        
        # 10. 样本量分布
        ax = axes[2, 2]
        sample_counts = self.analysis_results['subject_stats']['sample_counts']
        ax.hist(sample_counts, bins=15, alpha=0.7, color='lightcoral', edgecolor='black')
        ax.axvline(np.mean(sample_counts), color='red', linestyle='--',
                  label=f'平均: {np.mean(sample_counts):.0f}')
        ax.axvline(np.median(sample_counts), color='green', linestyle='--',
                  label=f'中位数: {np.median(sample_counts):.0f}')
        ax.set_title('受试者样本量分布', fontweight='bold')
        ax.set_xlabel('样本量')
        ax.set_ylabel('受试者数量')
        ax.legend()
        
        # 11. 综合决策雷达图
        ax = axes[2, 3]
        categories = ['差异显著性', '模式线性度', '受试者相似性', '特征异质性', 
                     '可分离性', '类别一致性', '样本充足性']
        
        scores = [
            self.decision_scores['phase1']['difference_significance'],
            self.decision_scores['phase1']['pattern_linearity'], 
            self.decision_scores['phase1']['subject_similarity'],
            self.decision_scores['phase1']['feature_heterogeneity'],
            self.decision_scores['phase2']['subject_separability'],
            self.decision_scores['phase2']['class_consistency'],
            self.decision_scores['phase3']['sample_adequacy']
        ]
        
        # 雷达图
        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
        scores_plot = scores + [scores[0]]  # 闭合图形
        angles_plot = np.concatenate((angles, [angles[0]]))
        
        ax.plot(angles_plot, scores_plot, 'o-', linewidth=2, color='blue')
        ax.fill(angles_plot, scores_plot, alpha=0.25, color='blue')
        ax.set_xticks(angles)
        ax.set_xticklabels(categories, fontsize=10)
        ax.set_ylim(0, 1)
        ax.set_title('Subject Embedding 可行性雷达图', fontweight='bold')
        ax.grid(True)
        
        plt.tight_layout()
        viz_path = os.path.join(self.save_path, 'visualizations', 'comprehensive_analysis.png')
        plt.savefig(viz_path, dpi=300, bbox_inches='tight')
        plt.show()
        
        print(f"✅ 可视化图表已保存: {viz_path}")
    
    def phase4_decision_generation(self):
        """
        Phase 4: 基于所有分析结果生成最终决策和建议
        """
        print("\n" + "="*80)
        print("🎯 Phase 4: 决策建议生成")
        print("="*80)
        
        # 综合所有决策得分
        all_scores = {}
        for phase in self.decision_scores:
            all_scores.update(self.decision_scores[phase])
        
        # 决策权重
        weights = {
            'difference_significance': 0.15,
            'pattern_linearity': 0.15,
            'subject_separability': 0.20,
            'class_consistency': 0.15,
            'sample_adequacy': 0.15,
            'embedding_feasibility': 0.20
        }
        
        # 计算加权总分
        weighted_score = sum(all_scores.get(key, 0) * weight 
                           for key, weight in weights.items())
        
        # 生成具体建议
        recommendations = []
        embedding_dim_suggestion = 32  # 默认维度
        
        # 基于各项指标生成建议
        if all_scores.get('difference_significance', 0) > 0.7:
            recommendations.append("✅ 受试者间差异显著，embedding很有必要")
        elif all_scores.get('difference_significance', 0) > 0.3:
            recommendations.append("📊 受试者间存在一定差异，embedding有帮助")
        else:
            recommendations.append("⚠️ 受试者间差异较小，embedding效果可能有限")
        
        if all_scores.get('pattern_linearity', 0) > 0.8:
            recommendations.append("✅ 差异模式高度线性，适合简单Subject Embedding")
            embedding_dim_suggestion = min(64, max(16, int(len(self.data['available_subjects']) * 1.5)))
        elif all_scores.get('pattern_linearity', 0) > 0.6:
            recommendations.append("📊 差异模式较为线性，Subject Embedding适用")
            embedding_dim_suggestion = min(128, max(32, int(len(self.data['available_subjects']) * 2)))
        else:
            recommendations.append("⚠️ 差异模式复杂，考虑非线性embedding或其他方法")
            embedding_dim_suggestion = min(256, max(64, int(len(self.data['available_subjects']) * 3)))
        
        if all_scores.get('subject_separability', 0) > 0.8:
            recommendations.append("⚠️ 受试者高度可分离，存在特征竞争风险，建议特征去偏差")
        elif all_scores.get('subject_separability', 0) > 0.5:
            recommendations.append("📊 受试者中等可分离，需要平衡embedding设计")
        
        if all_scores.get('sample_adequacy', 0) > 0.7:
            recommendations.append("✅ 样本量充足，支持复杂embedding方法")
        elif all_scores.get('sample_adequacy', 0) > 0.4:
            recommendations.append("📊 样本量中等，建议使用中等复杂度embedding")
        else:
            recommendations.append("⚠️ 样本量偏少，建议简单embedding方法")
        
        # 聚类分析建议
        if 'linearity_analysis' in self.analysis_results:
            best_silhouette = self.analysis_results['linearity_analysis']['best_silhouette']
            if best_silhouette > 0.5:
                recommendations.append("🔍 受试者形成明显聚类，可考虑Mixture of Experts")
            elif best_silhouette > 0.3:
                recommendations.append("📊 受试者有一定聚类趋势，可考虑分层embedding")
        
        # 最终决策
        if weighted_score > 0.7:
            final_decision = "强烈推荐使用Subject Embedding"
            confidence = "高"
        elif weighted_score > 0.5:
            final_decision = "建议使用Subject Embedding"
            confidence = "中"
        elif weighted_score > 0.3:
            final_decision = "可以尝试Subject Embedding"
            confidence = "低"
        else:
            final_decision = "不建议使用Subject Embedding"
            confidence = "高"
        
        # 具体实施建议
        implementation_suggestions = []
        
        if final_decision.startswith("强烈推荐") or final_decision.startswith("建议"):
            implementation_suggestions.extend([
                f"推荐embedding维度: {embedding_dim_suggestion}",
                "优先考虑可学习embedding层",
                "使用Adam优化器，学习率1e-4到1e-3",
                "添加embedding正则化防止过拟合"
            ])
            
            if all_scores.get('subject_separability', 0) > 0.6:
                implementation_suggestions.append("考虑特征去偏差或对抗训练")
            
            if 'linearity_analysis' in self.analysis_results:
                structure_correlation = self.analysis_results['linearity_analysis']['structure_correlation']
                if structure_correlation < 0.7:
                    implementation_suggestions.append("考虑非线性embedding (MLP)")
                else:
                    implementation_suggestions.append("线性embedding即可满足需求")
        
        # 保存决策结果
        decision_result = {
            'weighted_score': weighted_score,
            'final_decision': final_decision,
            'confidence': confidence,
            'recommendations': recommendations,
            'implementation_suggestions': implementation_suggestions,
            'embedding_dim_suggestion': embedding_dim_suggestion,
            'detailed_scores': all_scores
        }
        
        self.analysis_results['final_decision'] = decision_result
        
        # 打印决策报告
        print(f"\n📋 Subject Embedding 可行性分析报告")
        print("=" * 60)
        print(f"🎯 最终决策: {final_decision}")
        print(f"🎲 置信度: {confidence}")
        print(f"📊 综合得分: {weighted_score:.3f} / 1.0")
        
        print(f"\n💡 主要发现:")
        for rec in recommendations:
            print(f"  {rec}")
        
        if implementation_suggestions:
            print(f"\n🔧 实施建议:")
            for suggestion in implementation_suggestions:
                print(f"  • {suggestion}")
        
        print(f"\n📈 详细得分:")
        for key, score in all_scores.items():
            print(f"  - {key}: {score:.3f}")
        
        return decision_result
    
    def generate_report(self):
        """生成完整的分析报告"""
        report_path = os.path.join(self.save_path, 'subject_embedding_analysis_report.txt')
        
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("Subject Embedding 可行性分析报告\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"生成时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            # 数据概况
            f.write("📊 数据概况\n")
            f.write("-" * 30 + "\n")
            f.write(f"受试者数量: {len(self.data['available_subjects'])}\n")
            f.write(f"特征维度: {self.data['X_train'].shape[1]}\n")
            f.write(f"训练样本量: {len(self.data['X_train']):,}\n")
            f.write(f"验证样本量: {len(self.data['X_val']):,}\n")
            f.write(f"测试样本量: {len(self.data['X_test']):,}\n")
            
            # Phase 1 结果
            f.write(f"\n🔍 Phase 1: 受试者间差异分析\n")
            f.write("-" * 30 + "\n")
            if 'subject_stats' in self.analysis_results:
                sample_counts = self.analysis_results['subject_stats']['sample_counts']
                f.write(f"平均每受试者样本量: {np.mean(sample_counts):.0f}\n")
                f.write(f"样本量范围: [{np.min(sample_counts):.0f}, {np.max(sample_counts):.0f}]\n")
                
            if 'feature_variation' in self.analysis_results:
                pca_3pc = np.sum(self.analysis_results['feature_variation']['pca_explained_variance'][:3])
                f.write(f"前3个主成分解释方差: {pca_3pc*100:.1f}%\n")
                
            if 'subject_similarity' in self.analysis_results:
                corr_matrix = self.analysis_results['subject_similarity']['correlation_matrix']
                mean_corr = np.mean(corr_matrix[np.triu_indices_from(corr_matrix, k=1)])
                f.write(f"受试者间平均相关性: {mean_corr:.3f}\n")
            
            # Phase 2 结果
            f.write(f"\n🔍 Phase 2: 受试者可分离性评估\n")
            f.write("-" * 30 + "\n")
            if 'subject_identification' in self.analysis_results:
                accuracy = self.analysis_results['subject_identification']['accuracy']
                f.write(f"受试者识别准确率: {accuracy:.3f}\n")
                
            if 'class_consistency' in self.analysis_results and self.analysis_results['class_consistency']:
                avg_cv = np.mean([info['mean_cv'] for info in self.analysis_results['class_consistency'].values()])
                f.write(f"脑区平均变异系数: {avg_cv:.3f}\n")
            
            # Phase 3 结果
            f.write(f"\n🔍 Phase 3: Embedding适配性评估\n")
            f.write("-" * 30 + "\n")
            if 'linearity_analysis' in self.analysis_results:
                structure_corr = self.analysis_results['linearity_analysis']['structure_correlation']
                best_silhouette = self.analysis_results['linearity_analysis']['best_silhouette']
                f.write(f"PCA vs t-SNE结构相关性: {structure_corr:.3f}\n")
                f.write(f"最佳聚类轮廓系数: {best_silhouette:.3f}\n")
                
            if 'sample_adequacy' in self.analysis_results:
                adequacy_score = self.analysis_results['sample_adequacy']['adequacy_score']
                f.write(f"样本充足性得分: {adequacy_score:.3f}\n")
            
            # 最终决策
            f.write(f"\n🎯 最终决策\n")
            f.write("-" * 30 + "\n")
            if 'final_decision' in self.analysis_results:
                decision = self.analysis_results['final_decision']
                f.write(f"决策: {decision['final_decision']}\n")
                f.write(f"置信度: {decision['confidence']}\n")
                f.write(f"综合得分: {decision['weighted_score']:.3f}\n")
                f.write(f"建议embedding维度: {decision['embedding_dim_suggestion']}\n\n")
                
                f.write("主要发现:\n")
                for rec in decision['recommendations']:
                    f.write(f"  {rec}\n")
                
                f.write(f"\n实施建议:\n")
                for suggestion in decision['implementation_suggestions']:
                    f.write(f"  • {suggestion}\n")
        
        print(f"✅ 完整报告已保存: {report_path}")
        return report_path

# ============================================================================
# 🚀 主执行函数
# ============================================================================

def run_subject_embedding_analysis(X_train_scaled, y_train, X_val_scaled, y_val, 
                                 X_test_scaled, test_labels, save_path='./subject_embedding_analysis/'):
    """
    运行完整的Subject Embedding可行性分析
    
    Args:
        X_train_scaled: 标准化训练特征 (n_samples, 341)
        y_train: 训练标签 (n_samples, 102) one-hot或类别索引
        X_val_scaled: 标准化验证特征
        y_val: 验证标签
        X_test_scaled: 标准化测试特征  
        test_labels: 测试标签
        save_path: 结果保存路径
    
    Returns:
        analyzer: 分析器对象，包含所有结果
        decision_result: 最终决策结果
    """
    
    print("🧠 开始Subject Embedding可行性分析...")
    print("="*80)
    
    # 初始化分析器
    analyzer = SubjectEmbeddingAnalyzer(save_path=save_path)
    
    # Phase 0: 数据准备
    data = analyzer.prepare_data_with_subjects(
        X_train_scaled, y_train, X_val_scaled, y_val, X_test_scaled, test_labels
    )
    
    if data is None:
        print("❌ 数据准备失败，分析终止")
        return None, None
    
    # Phase 1: 受试者间差异分析
    analyzer.phase1_subject_differences_analysis()
    
    # Phase 2: 受试者可分离性评估
    analyzer.phase2_subject_separability_analysis()
    
    # Phase 3: Embedding适配性评估
    analyzer.phase3_embedding_adaptability_analysis()
    
    # 生成可视化
    analyzer.generate_visualizations()
    
    # Phase 4: 决策生成
    decision_result = analyzer.phase4_decision_generation()
    
    # 生成报告
    analyzer.generate_report()
    
    print("\n" + "="*80)
    print("🎉 Subject Embedding可行性分析完成!")
    print("="*80)
    print(f"📁 所有结果保存在: {save_path}")
    print(f"🎯 最终建议: {decision_result['final_decision']}")
    print(f"📊 综合得分: {decision_result['weighted_score']:.3f}")
    
    return analyzer, decision_result

# ============================================================================
# 💡 使用示例
# ============================================================================

# 假设你已经有了处理好的数据
analyzer, decision = run_subject_embedding_analysis(
    X_train_scaled=X_train_scaled,     # 你的训练数据 
    y_train=y_train,                   # 你的训练标签
    X_val_scaled=X_val_scaled,         # 你的验证数据
    y_val=y_val,                       # 你的验证标签
    X_test_scaled=X_test_scaled,       # 你的测试数据
    test_labels=test_labels,           # 你的测试标签
    save_path='./subject_embedding_analysis/'
)

# 查看决策结果
print("最终建议:", decision['final_decision'])
print("建议embedding维度:", decision['embedding_dim_suggestion'])
print("实施建议:", decision['implementation_suggestions'])

# 访问详细分析结果
print("受试者识别准确率:", analyzer.analysis_results['subject_identification']['accuracy'])
print("PCA前3PC解释方差:", np.sum(analyzer.analysis_results['feature_variation']['pca_explained_variance'][:3]))
